# Day 061 — Exercise 5: Health Report

A **health endpoint** is the first thing load balancers, Kubernetes, and on-call engineers check when something goes wrong. A good health endpoint doesn't just return `{"status": "ok"}` — it checks each dependency (database, cache, model server) and reports which ones are healthy.

Three status values:

| Status | Meaning |
|--------|--------|
| `ok` | All dependencies healthy |
| `degraded` | Some healthy, some not — service partially functional |
| `down` | All dependencies unhealthy — service not functional |

`degraded` is important: it lets the load balancer keep sending traffic (some requests can still succeed) while alerting the team to investigate.

In [ ]:
from typing import Callable


## Task

Implement `make_health_report(services: dict[str, Callable[[], bool]]) -> dict`:

- Call each `check_fn()` — wrap in try/except, treat raised exceptions as `False`
- `status = 'ok'` if all pass, `'down'` if all fail, `'degraded'` if mixed
- Return `{'status': ..., 'services': {name: bool, ...}}`
- Empty `services` dict → `'ok'`

## Your Implementation

In [ ]:
def make_health_report(services: dict[str, Callable[[], bool]]) -> dict:
    """Run each service check and return an aggregated health report.

    services: {name: check_fn} where check_fn() returns True (healthy) / False (unhealthy).
              check_fn() may also raise — treat a raised exception as False.

    Returns:
    {
        "status": "ok" | "degraded" | "down",
        "services": {name: bool, ...},
    }

    Status rules:
        "ok"       — all checks pass
        "degraded" — at least one passes and at least one fails
        "down"     — all checks fail (or no checks provided → "ok")
    """
    # TODO: run checks, collect results, compute status, return dict
    raise NotImplementedError


In [ ]:
def make_health_report(services: dict[str, Callable[[], bool]]) -> dict:
    results = {}
    for name, fn in services.items():
        try:
            results[name] = bool(fn())
        except Exception:
            results[name] = False

    if not results:
        status = "ok"
    elif all(results.values()):
        status = "ok"
    elif not any(results.values()):
        status = "down"
    else:
        status = "degraded"

    return {"status": status, "services": results}


## Automated checks

In [ ]:
score, total = 0, 6
try:
    # all ok
    r1 = make_health_report({"db": lambda: True, "cache": lambda: True})
    assert r1["status"] == "ok", f"Expected 'ok', got {r1['status']!r}"
    assert r1["services"] == {"db": True, "cache": True}
    score += 1; print("\u2705 all passing checks → status='ok'")

    # all failing
    r2 = make_health_report({"db": lambda: False, "cache": lambda: False})
    assert r2["status"] == "down", f"Expected 'down', got {r2['status']!r}"
    score += 1; print("\u2705 all failing checks → status='down'")

    # mixed → degraded
    r3 = make_health_report({"db": lambda: True, "cache": lambda: False})
    assert r3["status"] == "degraded", f"Expected 'degraded', got {r3['status']!r}"
    assert r3["services"]["db"] is True
    assert r3["services"]["cache"] is False
    score += 1; print("\u2705 mixed checks → status='degraded'")

    # exception in check counts as False
    def bad_check():
        raise RuntimeError("connection refused")

    r4 = make_health_report({"db": lambda: True, "cache": bad_check})
    assert r4["services"]["cache"] is False
    assert r4["status"] == "degraded"
    score += 1; print("\u2705 exception in check_fn treated as False")

    # empty services → ok
    r5 = make_health_report({})
    assert r5["status"] == "ok"
    score += 1; print("\u2705 empty services dict → status='ok'")

    # single failing → down
    r6 = make_health_report({"only": lambda: False})
    assert r6["status"] == "down"
    score += 1; print("\u2705 single failing check → status='down'")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def make_health_report(services: dict[str, Callable[[], bool]]) -> dict:
    results = {}
    for name, fn in services.items():
        try:
            results[name] = bool(fn())
        except Exception:
            results[name] = False

    if not results:
        status = "ok"
    elif all(results.values()):
        status = "ok"
    elif not any(results.values()):
        status = "down"
    else:
        status = "degraded"

    return {"status": status, "services": results}
```

**Why wrap in try/except?** A check_fn that tries to connect to a database or ping a service can always raise (network timeout, DNS failure, auth error). A crashed check_fn doesn't mean the health endpoint itself should 500 — it means that specific dependency is down. Catch broadly and report `False`.

</details>